# 01 — Getting Started with `midas_calibrate_v2`

This is the entry point.  By the end you will have:

1. Loaded a real synchrotron calibrant image (Varex 4343CT, CeO₂ at 63 keV).
2. Built a v2 calibration spec from a paramstest file.
3. Run the production `autocalibrate_pv` pipeline.
4. Read out the converged geometry and per-parameter Bayesian σ.
5. Run the three reliability gates that decide whether the calibration should be trusted.

The whole thing takes ~30–60 seconds on a CPU.

## Pre-flight: where is the test data?

The four reference calibrant datasets live under `V2_TEST_BASE`
(default `/tmp/midas_v2_test`).  The Varex CeO₂ dataset is the
canonical example — every notebook in this directory uses it.


In [1]:
import os, sys
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

from pathlib import Path
BASE = Path(os.environ.get('V2_TEST_BASE', '/tmp/midas_v2_test'))
PARAMS = BASE / 'refined_MIDAS_params_Ceria_63keV_900mm_100x100_0p5s_aero_0.txt'
IMAGE  = BASE / 'Ceria_63keV_900mm_100x100_0p5s_aero_0_001137.tif'

assert PARAMS.exists(), f'paramstest not found at {PARAMS} — set $V2_TEST_BASE'
assert IMAGE.exists(), f'image not found at {IMAGE}'
print('OK:', BASE)


OK: /tmp/midas_v2_test


## Step 1 — Load the detector image

MIDAS convention is detector-Y horizontal, detector-Z vertical.  Most
synchrotron TIFFs need a vertical flip (`[::-1, :]`) to match — the
`.copy()` call sidesteps a `torch.as_tensor` warning about negative
strides.


In [2]:
import numpy as np
from PIL import Image

image = np.array(Image.open(str(IMAGE))).astype(np.float64)[::-1, :].copy()
print(f'image: shape={image.shape}, dtype={image.dtype}, '
      f'min={image.min():.0f}, max={image.max():.0f}, mean={image.mean():.0f}')


image: shape=(2880, 2880), dtype=float64, min=0, max=8092, mean=39


## Step 2 — Build the v2 calibration spec

`spec_from_v1_file` reads a v1 paramstest and creates a tree of v2
`Parameter` objects.  Each parameter knows whether it is refined,
its bounds, and (later) any prior you want to add.

The v1 paramstest defines the seed values for L_sd, BC, tilts,
distortion harmonics, and the calibrant material/wavelength.


In [3]:
from midas_calibrate.params import CalibrationParams as V1Params
from midas_calibrate_v2.compat.from_v1 import spec_from_v1_file

v1 = V1Params.from_file(PARAMS)
# small fix-ups required by the cake/ring builder for some old paramstest files
if v1.RBinSize <= 0 or v1.RBinSize > 0.5:
    v1.RBinSize = 0.25
if v1.EtaBinSize <= 0:
    v1.EtaBinSize = 5.0
v1.MaxRingRad = max(v1.MaxRingRad, v1.RhoD / max(v1.pxY, 1.0))
v1.Width = max(v1.Width, 800.0)

spec = spec_from_v1_file(PARAMS)
print(f'spec: {len(spec.parameters)} total parameters, '
      f'{len(spec.refined_names())} refined by default')
print('refined names:', spec.refined_names())


spec: 26 total parameters, 20 refined by default
refined names: ['Lsd', 'BC_y', 'BC_z', 'ty', 'tz', 'a2', 'a4', 'iso_R2', 'phi4', 'iso_R6', 'iso_R4', 'phi2', 'a1', 'phi1', 'a3', 'phi3', 'a5', 'phi5', 'a6', 'phi6']


## Step 3 — Run the production pipeline

`autocalibrate_pv` is the production single-image entry point.  It
alternates an E-step (cake + pseudo-Voigt peak fitting) with an
M-step (Levenberg-Marquardt minimisation of the pseudo-strain
residual).

`reuse_fits=True` is the recommended setting for clean calibrant
images — it does the cake+pV fit ONCE on iter 0 and re-uses the
fitted (Y, Z) positions across the LM iterations.  This avoids
re-extract noise destabilising the LM convergence.


In [4]:
import time
from midas_calibrate_v2.pipelines.single_pv import autocalibrate_pv

t0 = time.time()
res = autocalibrate_pv(
    v1, image, spec=spec,
    n_iter=4,                            # 4 outer iterations
    half_window_px=8.0,                   # peak-fit half-window
    snr_min=8.0,                          # drop fits with SNR < 8
    trim_mode='stratified_multfactor',     # spatial-aware outlier rejection
    trim_residual_pct=5.0,                # MultFactor 5x MAD per cell
    reuse_fits=True,
    lm_max_iter=300,
    verbose=False,
    distribution_report=False,
)
elapsed = time.time() - t0
final = res.history[-1]
print(f'elapsed: {elapsed:.1f} s')
print(f'final strain: {final.mean_strain_uE:.3f} µε')
print(f'L_sd: {final.Lsd:.2f} µm')
print(f'BC:   ({final.BC_y:.4f}, {final.BC_z:.4f}) px')
print(f'tilts: ty={final.ty:+.4f}°  tz={final.tz:+.4f}°')
print(f'fits used: {res.fits_final.Y_pix.numel()}')


/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  return torch.sparse_csr_tensor(indptr, indices, values,
/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:51.)
  return torch.sparse_csr_tensor(indptr, in

elapsed: 36.3 s
final strain: 7.738 µε
L_sd: 895925.10 µm
BC:   (1446.9748, 1468.9109) px
tilts: ty=-0.3082°  tz=+0.3885°
fits used: 4532


## Step 4 — Per-parameter Bayesian σ via Laplace

The MAP estimate above is the headline geometry.  But `midas_calibrate_v2`
also gives you the **Cramér-Rao σ** on every refined parameter — for
free, from the same autograd graph the LM uses.

Build a closure that mirrors what the LM saw, then call
`fisher_at_map`.  The result is a `LaplaceResult` whose
`sigma_per_dim` field carries the per-parameter σ in physical units
(µm for L_sd, px for BC, deg for tilts, dimensionless for distortion
harmonics).


In [5]:
import torch
from midas_calibrate_v2.loss.pseudo_strain import pseudo_strain_residual
from midas_calibrate_v2.inference.laplace import fisher_at_map

fits = res.fits_final

def residual_fn(unp):
    return pseudo_strain_residual(
        fits.Y_pix, fits.Z_pix, fits.ring_two_theta_deg, unp,
        rho_d=fits.rho_d, weights=fits.weights,
        ring_idx=fits.ring_idx,
        ring_d_spacing_A=fits.ring_d_spacing_A,
    )

with torch.no_grad():
    r = residual_fn(res.unpacked)
    sigma_r = float(((r * r).mean()) ** 0.5)
print(f'empirical residual σ_r: {sigma_r:.3e}')

lap = fisher_at_map(spec, residual_fn, res.unpacked,
                    sigma_r=sigma_r, ridge=1e-9,
                    dtype=torch.float64, device='cpu')

# Expand vector parameters into per-element names for readability
def _flat(lap):
    out = []
    for n, o, s in zip(lap.refined_names, lap.refined_offsets, lap.refined_sizes):
        for k in range(s):
            out.append(f'{n}[{k}]' if s > 1 else n)
    return out

flat = _flat(lap)
sigma_arr = lap.sigma_per_dim.detach().cpu().numpy()
HEAD = ('Lsd', 'BC_y', 'BC_z', 'ty', 'tz')
print('\nHeadline geometry σ:')
for nm, s in zip(flat, sigma_arr):
    if nm in HEAD:
        unit = 'µm' if nm == 'Lsd' else ('px' if nm.startswith('BC') else 'deg')
        print(f'  σ({nm:<6s}) = {s:.4e} {unit}')


empirical residual σ_r: 1.049e-05

Headline geometry σ:
  σ(Lsd   ) = 7.1737e-01 µm
  σ(BC_y  ) = 3.3425e-04 px
  σ(BC_z  ) = 3.2261e-04 px
  σ(ty    ) = 3.1305e-04 deg
  σ(tz    ) = 3.4464e-04 deg


## Step 5 — The three reliability gates

A converged calibration with low pseudo-strain is necessary but not
sufficient.  Three gates catch the common failure modes:

- **Strain-cap** — reject if mean ε > 100 µε (catches LM basin escape)
- **Basin-check** — warn if MAP drifted too far from seed
- **Cross-validation** — train on rings 0..N−1, test on rings ≥ N (catches misspecified distortion basis)


In [6]:
strain_uE = res.history[-1].mean_strain_uE
print(f'mean pseudo-strain: {strain_uE:.2f} µε')
if strain_uE > 100.0:
    print('  ✗ STRAIN-CAP FAILED — rejected as basin escape')
else:
    print('  ✓ strain-cap passed')

seed_lsd = float(spec.parameters['Lsd'].init)
final_lsd = float(res.unpacked['Lsd'])
seed_BC_y = float(spec.parameters['BC_y'].init)
final_BC_y = float(res.unpacked['BC_y'])
drift_lsd_pct = abs(final_lsd / seed_lsd - 1.0) * 100
drift_BC_y_px = abs(final_BC_y - seed_BC_y)
print(f'\nL_sd drift: {drift_lsd_pct:.4f}% (basin width: 0.3%)')
print(f'BC_y drift: {drift_BC_y_px:.3f} px (basin width: 1.5 px)')
if drift_lsd_pct > 0.3 or drift_BC_y_px > 1.5:
    print('  ⚠  BASIN-CHECK WARNING — verify by hand')
else:
    print('  ✓ basin check passed')


mean pseudo-strain: 7.74 µε
  ✓ strain-cap passed

L_sd drift: 0.0000% (basin width: 0.3%)
BC_y drift: 0.000 px (basin width: 1.5 px)
  ✓ basin check passed


For the **cross-validation gate**, see `runners/run_cross_validation.py`
in the dev tree — the gate code is in `pipelines/diagnostics.py`.
On the Varex CeO₂ headline above, the CV gate notably FIRES at
+95 % held-out-ring systematic with KS p < 10⁻³⁰: the standard
15-coefficient distortion basis is incomplete on the outer rings
even of an "ideal" calibrant.  See notebook **05** for the full
diagnosis and the per-ring `δr_k` fix.


## Where to next

- **Notebook 02** — Bayesian uncertainty: full Laplace + Fisher; σ(Q) propagation for downstream PDF/Rietveld.
- **Notebook 03** — Multi-panel Pilatus3 2M-CdTe with the Σ=0 gauge.
- **Notebook 04** — Refining pixel size and wavelength (the "you need an external prior" recipes).
- **Notebook 05** — Reliability gates + the per-ring DC fix.

For an exhaustive set of analysis runners, see `dev/paper/runners/`
in this repository.
